In [0]:
# Cria um campo de seleção no topo do notebook
dbutils.widgets.dropdown("environment", "DEV", ["DEV", "PROD"])
env = dbutils.widgets.get("environment").lower() # Retorna "dev" ou "prod"

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Phase 1 - Ingest Open-Meteo
# MAGIC
# MAGIC Fetch one day of hourly weather data for the configured cities.

# COMMAND ----------

from urllib.request import urlopen
from urllib.parse import urlencode
import json
from datetime import datetime, timezone

from pyspark.sql import Row

cities = [
    {"name": "Brasilia", "latitude": -15.793889, "longitude": -47.882778},
    {"name": "Sao Paulo", "latitude": -23.55052, "longitude": -46.633308},
    {"name": "Rio de Janeiro", "latitude": -22.906847, "longitude": -43.172896},
]

rows = []

for city in cities:
    params = urlencode({
        "latitude": city["latitude"],
        "longitude": city["longitude"],
        "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m,precipitation",
        "forecast_days": 1,
        "timezone": "America/Sao_Paulo",
    })
    with urlopen(f"https://api.open-meteo.com/v1/forecast?{params}", timeout=30) as response:
        payload = json.loads(response.read().decode("utf-8"))

    rows.append({
        "city": city["name"],
        "latitude": city["latitude"],
        "longitude": city["longitude"],
        "hourly": payload["hourly"],
        "ingestion_timestamp": datetime.now(timezone.utc).isoformat(),
    })

raw_df = spark.createDataFrame(rows)
#display(raw_df)

# Lendo os dados persistidos dinamicamente usando os parâmetros que você configurou
catalog_name = dbutils.widgets.get("environment") # ex: "dev" ou "prod"

# escrever os dados na tabela
#raw_df.write.mode("overwrite").saveAsTable(f"{catalog_name}.bronze_db.open_meteo_raw")

